### Preprocessing for Better Retrieval (tax.docx -> tax_with_table.docx)
- The query-related content in the original document was stored as an image, which the LLM cannot read. To fix this, the image was converted into a table within the document, resulting in the new file: tax_with_table.docx

In [ ]:
# %pip install --upgrade -q docx2txt langchain-community

In [ ]:
# %pip install -qU langchain-text-splitters

In [ ]:
# 1,2 : read&load document -> split document

from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader("./tax_with_table.docx")
# split document
document_list = loader.load_and_split(text_splitter=text_splitter)

In [ ]:
len(document_list)

In [ ]:
# pull text embedding model (text to vector)
# !ollama pull nomic-embed-text

In [ ]:
# 3: Embedding
from langchain_ollama import OllamaEmbeddings

embedding = OllamaEmbeddings(model="nomic-embed-text")

# dimension 확인용 test
# test = embedding.embed_query("test")
# print(len(test))  # output을 pinecone index의 dimensions에 넣어

In [ ]:
# %pip install --upgrade --quiet \
#     langchain-pinecone \
#     langchain-ollama \
#     langchain

In [ ]:
from dotenv import load_dotenv
import os
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

# pinecone 연결
load_dotenv()
pinecone_api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=pinecone_api_key)

# pinecone index에 document_list를 embedding
database = PineconeVectorStore.from_documents(
    document_list, embedding, index_name="tax-table-index"
)

In [ ]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"

In [ ]:
# 5: Pass retrieved documents along with the question to the LLM
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3:latest")

In [ ]:
# RetrievalQA Chain
# %pip install -U langchain langchainhub -q

In [ ]:
from langchain_classic import hub

# get prompt from hub
prompt = hub.pull("rlm/rag-prompt")

- Even after converting the image to a table in the docx, retrieval still failed to find the relevant document
- Retrieval should find Article 55, but it returns irrelevant sections instead, resulting in an incorrect answer to the user.

In [ ]:
# docx에서 query에 해당하는 부분을 image -> table로 변경했음에도 관련 문서로 인식 실패
retriever = database.as_retriever()
retriever.invoke(query)

In [ ]:
# Build QA Chain
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm, retriever=retriever, chain_type_kwargs={"prompt": prompt}
)
ai_message = qa_chain.invoke({"query": query})

In [ ]:
ai_message